# 1. Data

> **Runtime: CPU.** Runtime → Change runtime type → **CPU**. No GPU needed: downloads, Parquet conversion, preparation.

Raw data → Parquet (download if missing) → hourly series, splits, window parameters. Kaggle datasets (PJM, Bitcoin) need `~/.kaggle/kaggle.json` — put it at `REPO/kaggle.json` (the cell below copies it).

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# ── setup: Drive, code, dependencies, paths ──
REPO = "/content/drive/MyDrive/Colab Notebooks/MarathonTS"   # project folder on Drive (or a git clone)
PROFILE = "min"            # "min": the Colab run (fast); "mid": min + the VAE per seed; "max": everything on
# results go to {REPO}/artifacts/<PROFILE>; set ARTIFACTS = "..." to change
exec(open(f"{REPO}/colab/colab_setup.py").read())

In [ ]:
# Kaggle credentials (only if data/raw has no PJM / Bitcoin yet)
import os, shutil
if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")) and os.path.exists(f"{REPO}/kaggle.json"):
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    shutil.copy(f"{REPO}/kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))

In [ ]:
# ── scope of this run: change freely; finished units are skipped on re-run ──
import dataclasses
cfg = dataclasses.replace(
    cfg,
    datasets=list(cfg.datasets),  # the profile's datasets (marathonts/bench/config.py DATASETS); or a list, e.g. ["ett", "my_ds"]
    methods=None,                # None → the suites on every pool + long_methods; or a list to run a subset
    baselines=None,              # None → long_methods (case study: CS9, CS10)
    # custom components (EXTENDING.md): plugin files + one line each, no code changes:
    # plugins=["my_plugins.py"],                          # files registering them (path from the project folder)
    # pool_suites={"vae": "tsgm_vae", "gan": "my_gan"},   # the whole suite on another pool generator → gan_zseries, …
    # extra_assemblers=["my_stitch"],                     # a stitcher on every pool → vae_my_stitch, …
    # extra_joiners=["my_smooth"],                        # your own seam processing (run like the built-in ones)
    # smooth_paths=["zseries", "greedy"],                 # stitchers that get seam processing ("all": all but the random path and vae_quilting; min: ["zseries"])
    # joiners=["crossfade", "wsola"],                     # which seam processings (None: all)
    # disabled=["pctsp", "chronos"],                      # switch off methods or components by name
    # noise_twin=True,                                    # + the noise twin vae_rand_b (min: off; mid, max: on)
    # frozen_methods=[],                                  # published results kept as they are, not run (default: vae_zseries and its
    #                                                     #   seam-processing variants, marathonts/bench/config.py FROZEN_METHODS); [] → left out of the tables
    # long_methods=[*cfg.long_methods, "my_long"],          # + a long-series generator (method and baseline)
    seeds=list(range(10)),
)
# plugins=[...] above loads custom components (datasets, generators, assemblers, joiners, models, metrics, tasks) in
# every notebook, in the stage subprocesses and in the hypothesis part — EXTENDING.md

In [ ]:
from marathonts.bench.run import stage_raw, stage_prepare
stage_raw(cfg)
stage_prepare(cfg)

In [ ]:
import pandas as pd
from marathonts.bench.data import load_prepared, window_layout
metas = [load_prepared(ds, cfg).meta for ds in cfg.datasets]
pd.DataFrame([{**m, **window_layout(m, cfg)} for m in metas])[["name", "n", "n_train", "K", "N", "start", "end"]]